In [ ]:
!pip install faster-whisper
!apt-get install ffmpeg -y
!pip install pyngrok
!pip install rapidfuzz, dotenv
!pip install webrtcvad

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [9]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from faster_whisper import WhisperModel
import webrtcvad

In [ ]:

# Set your ngrok authtoken

from pyngrok import ngrok
from dotenv import load_dotenv

load_dotenv()
ngrok.set_auth_token(GNORK_AUTH_TOKEN)


In [4]:
model = WhisperModel("medium", compute_type="int8")  # Use "cpu" if needed

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


vocabulary.txt:   0%|          | 0.00/460k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.26k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

model.bin:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

In [5]:
# Load model

# model = WhisperModel("medium", device = 'cuda', compute_type="float32")


In [10]:
# WebRTC VAD setup
vad = webrtcvad.Vad()
vad.set_mode(2)  # 0 = least aggressive, 3 = most aggressive

In [12]:
!pip install pydub


In [13]:
from pydub import AudioSegment

In [26]:
# Function: Remove silence from audio chunk

def remove_silence(audio: AudioSegment, frame_ms=30):
    audio = audio.set_channels(1).set_frame_rate(16000)
    frame_len = int(16000 * frame_ms / 1000) * 2  # bytes for 16-bit PCM mono
    raw_audio = audio.raw_data

    speech_frames = []
    for i in range(0, len(raw_audio), frame_len):
        chunk = raw_audio[i:i + frame_len]
        if len(chunk) < frame_len:
            continue
        if vad.is_speech(chunk, 16000):
            speech_frames.append(chunk)

    # Combine speech frames
    speech_audio = b"".join(speech_frames)
    output = AudioSegment(
        data=speech_audio, sample_width=2, frame_rate=16000, channels=1
    )
    return output


In [40]:
# Create Flask app
# translator = Translator()
app = Flask(__name__)

@app.route("/transcribe", methods=["POST"])
def transcribe_audio():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400

    file = request.files['file']
      # Receive the form data
    lan_code = request.form.get('lan_code')  # lan_code sent via form
    TOPIC = request.form.get('prompt')      # prompt sent via form

    main_prompt = f"This is a video call on {TOPIC}."
    # Transcribe the conversation with attention to the topic-specific vocabulary, tone, and context. If it’s a business meeting, focus on professional terms, action items, and speaker roles. If it's casual, reflect informal language and small talk. For technical or academic discussions, prioritize accuracy in jargon and topic-relevant terms."

    if (lan_code == "en"):
      model_task = "transcribe"
    else:
      model_task = "translate"
    # model_task = "translate"
    file_path = "/content/temp.wav"
    file.save(file_path)

    # chunk = AudioSegment.from_file(file_path)

    # print("🧹 Removing silence...")
    # trimmed = remove_silence(chunk)

    segments, _ = model.transcribe(file_path, initial_prompt = main_prompt, language=lan_code, task=model_task , vad_filter = True)

    result = " ".join([segment.text for segment in segments])

    corrected_text = correct_transcription(result)


    return jsonify({"translation": corrected_text})




In [28]:
from rapidfuzz import fuzz, process

# List of names you want the model to recognize
KNOWN_NAMES = ["Ankit", "Aman", "Asmit", "Dhaubanjar", "Pokhrel", "Phuyal", "SIC", "Pulchowk" ]

def correct_transcription(transcript: str, threshold: int = 60) -> str:
    words = transcript.split()
    corrected_words = []

    for word in words:
        match, score, _ = process.extractOne(word, KNOWN_NAMES, scorer=fuzz.ratio)
        if score >= threshold:
            corrected_words.append(match)
        else:
            corrected_words.append(word)

    return " ".join(corrected_words)


In [41]:
# Open ngrok tunnel
public_url = ngrok.connect(addr=5000)
print(f"🔥 Your public URL: {public_url}")

# Run the app
app.run(port=5000)

🔥 Your public URL: NgrokTunnel: "https://427a-35-197-6-142.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/May/2025 10:33:49] "POST /transcribe HTTP/1.1" 200 -


In [ ]:
fuzz.token_ratio("s i c", "sic")